# 02 ? Text Preprocessing

Apply the full NLP pipeline: cleaning, tokenisation, and sequence padding.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd
import config
from src.preprocessing import load_raw_data, clean_text, preprocess_dataframe

## 1. Load raw data

In [ ]:
df_raw = load_raw_data()
df_raw.head()

## 2. Demonstrate cleaning on sample messages

In [ ]:
samples = [
    'FREE entry in 2 a wkly comp to WIN FA Cup final tkts 21st May 2005!',
    'Hey, are you coming to the party tonight? Let me know!',
    'URGENT! You have won a ?1000 gift card. Call 09063442151 now!',
]
for s in samples:
    print(f'ORIGINAL : {s}')
    print(f'CLEANED  : {clean_text(s)}')
    print()

## 3. Full preprocessing

In [ ]:
df = preprocess_dataframe(df_raw)
print(df[['v2','label','cleaned_text']].head())
print(f"\nNull cleaned rows: {df['cleaned_text'].isna().sum()}")
print(f"Empty cleaned rows: {(df['cleaned_text']=='').sum()}")

## 4. Tokenisation

In [ ]:
from sklearn.model_selection import train_test_split
from src.tokenizer_utils import build_tokenizer, texts_to_padded_sequences, get_vocab_size

X = df['cleaned_text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
tokenizer = build_tokenizer(X_train.tolist())

X_train_seq = texts_to_padded_sequences(tokenizer, X_train.tolist())
X_test_seq  = texts_to_padded_sequences(tokenizer, X_test.tolist())

print("X_train shape:", X_train_seq.shape)
print("X_test shape :", X_test_seq.shape)
print("Vocab size   :", get_vocab_size(tokenizer))

## 5. Sequence length distribution

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
lengths = [len(tokenizer.texts_to_sequences([t])[0]) for t in X_train]
plt.figure(figsize=(8,4))
plt.hist(lengths, bins=50, color='#6c63ff', edgecolor='none')
plt.axvline(config.MAX_SEQ_LEN, color='red', linestyle='--', label=f'MAX_SEQ_LEN={config.MAX_SEQ_LEN}')
plt.title('Token Sequence Lengths (before padding)')
plt.xlabel('Length'); plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('../saved_plots/sequence_lengths.png', dpi=150)
plt.show()
print(f"95th percentile: {sorted(lengths)[int(len(lengths)*0.95)]}")